In [ ]:
import os
import base64
import uuid
from pathlib import Path
from io import BytesIO
import requests
from typing import List, Dict, Any

from dotenv import load_dotenv
from PIL import Image

# LangChain 1.0+ imports
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Docling for PDF parsing
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
from docling_core.types.doc import PictureItem

# OpenAI for image summarization
from openai import OpenAI as OpenAIClient

# Load environment variables
load_dotenv()

# Verify API key
if not os.getenv("OPEN_AI_API"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

In [ ]:
# Configuration
PDF_URL = "https://www.nlm.nih.gov/about/2014CJ_NLM.pdf"
PDF_PATH = "./data/2014CJ_NLM.pdf"
IMAGES_DIR = "./data/images"
CHROMA_DB_DIR = "./chroma_db"

# Create directories
Path("./data").mkdir(exist_ok=True)
Path(IMAGES_DIR).mkdir(exist_ok=True)

# Configure embeddings with local Qwen3
print("Loading Qwen3-Embedding-0.6B model from HuggingFace...")
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={'device': 'cpu', 'trust_remote_code': True},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 10}
)

# Initialize OpenAI client for image summarization
openai_client = OpenAIClient(api_key=os.getenv("OPEN_AI_API"))

In [ ]:
def download_pdf(url: str, save_path: str) -> str:
    """Download PDF from URL."""
    if os.path.exists(save_path):
        print(f"PDF already exists at {save_path}")
        return save_path
    
    print(f"Downloading PDF from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    
    with open(save_path, 'wb') as f:
        f.write(response.content)
    
    return save_path

def parse_pdf_with_docling(pdf_path: str, images_dir: str) -> tuple:
    """Parse PDF with Docling and extract text, tables, and images."""
    
    print("Parsing PDF with Docling...")
    
    # Configure pipeline to generate picture images
    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = 2.0
    pipeline_options.generate_picture_images = True
    pipeline_options.generate_page_images = False
    
    # Create converter with format options
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    # Convert the PDF
    result = converter.convert(pdf_path)
    
    # Get the document text
    doc_text = result.document.export_to_markdown()
    
    # Extract images with metadata
    images_data = []
    picture_counter = {}
    
    print("Extracting images from document...")
    for element, _level in result.document.iterate_items():
        if isinstance(element, PictureItem):
            page_num = element.prov[0].page_no if element.prov else 0
            
            if page_num not in picture_counter:
                picture_counter[page_num] = 0
            else:
                picture_counter[page_num] += 1
            
            filename = f"page_{page_num}_image_{picture_counter[page_num]}.png"
            filepath = os.path.join(images_dir, filename)
            image = element.get_image(result.document)
            
            if image:
                image.save(filepath, "PNG")                                
                images_data.append({
                    'ref': element.self_ref,
                    'page': page_num,
                    'local_path': filepath,
                    'filename': filename,
                    'position': element.prov[0].bbox.as_tuple() if element.prov else None
                })
                    
    return result, doc_text, images_data

pdf_path = download_pdf(PDF_URL, PDF_PATH)
docling_result, document_text, images_data = parse_pdf_with_docling(pdf_path, IMAGES_DIR)
print(f"{len(images_data)} images found in the document")

### **Summarize Images**

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    """Encode image to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def summarize_image(image_path: str, caption: str = "") -> str:
    """Summarize image using GPT-4.1-mini vision capabilities."""
    
    # Encode image
    base64_image = encode_image_to_base64(image_path)
    
    # Create prompt
    prompt = f"""Analyze all the details in this image, including any diagrams, graphs, or visual data representations. 
Your task is to provide a concise but comprehensive summary of the image (4-6 sentences) with as much detail as possible. 
Your response should include:
- A detailed description of the main focus or subject of the image.
- For any diagrams or graphs: what information they convey, a detailed description of the data, and any observed trends or conclusions that can be drawn.
- Any other detail or information that a human observer would find useful or relevant.
- Respond in complete sentences, and aim to provide a comprehensive and informative response.
- For any schemas, or flowcharts describe them in a way that a human reading your description could recreate the diagram.
- Any specific text that is shown in the image (with context).
If you are unable to summarize it, respond with an empty string. Do not respond with "I can't do that" or similar. 
"""
    
    # Call GPT-4.1-mini with vision
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
    )
    
    return response.choices[0].message.content

# Summarize all images
print("Summarizing images...\n")
for img in images_data:
    summary = summarize_image(img['local_path'], img.get('caption', ''))
    img['summary'] = summary

# Store for later use
saved_images = images_data
print("Example Image summary output")
print(images_data[0])

### **Create Text Chunks with Image Summaries**

In [ ]:
def create_chunks_with_images(doc_text: str, images: List[Dict], chunk_size: int = 1000) -> List[Dict]:
    """Create text chunks and insert image summaries at appropriate locations."""
    
    # Simple chunking strategy: split by paragraphs and combine to reach chunk_size
    paragraphs = doc_text.split('\n\n')
    chunks = []
    current_chunk = ""
    current_page = 1
    
    # Create a mapping of pages to images
    page_to_images = {}
    for img in images:
        page = img['page']
        if page not in page_to_images:
            page_to_images[page] = []
        page_to_images[page].append(img)
    
    for para in paragraphs:
        # Estimate page number (rough approximation)
        if len(current_chunk) + len(para) > chunk_size and current_chunk:
            # Save current chunk
            chunks.append({
                'text': current_chunk.strip(),
                'type': 'text',
                'page': current_page,
                'metadata': {'source': 'document_text'}
            })
            current_chunk = ""
            current_page += 1
        
        current_chunk += para + "\n\n"
        
        # Check if we should insert image summaries
        if current_page in page_to_images:
            for img in page_to_images[current_page]:
                # Add image summary as a chunk
                chunks.append({
                    'text': f"[IMAGE on page {img['page']}] {img['summary']}",
                    'type': 'image_summary',
                    'page': img['page'],
                    'metadata': {
                        'source': 'image',
                        'image_path': img['local_path'],
                        'image_filename': img['filename'],
                    }
                })
            # Clear images for this page
            del page_to_images[current_page]
    
    # Add remaining chunk
    if current_chunk.strip():
        chunks.append({
            'text': current_chunk.strip(),
            'type': 'text',
            'page': current_page,
            'metadata': {'source': 'document_text'}
        })
    
    # Add any remaining images
    for page, imgs in page_to_images.items():
        for img in imgs:
            chunks.append({
                'text': f"[IMAGE on page {img['page']}] {img['summary']}",
                'type': 'image_summary',
                'page': img['page'],
                'metadata': {
                    'source': 'image',
                    'image_path': img['local_path'],
                    'image_filename': img['filename'],
                }
            })
    
    return chunks

chunks = create_chunks_with_images(document_text, saved_images, chunk_size=1000)
print(f"Created {len(chunks)} chunks")
print(f"  - Text chunks: {sum(1 for c in chunks if c['type'] == 'text')}")
print(f"  - Image summary chunks: {sum(1 for c in chunks if c['type'] == 'image_summary')}")

### **Build ChromaDB Vector Stores with LangChain**

In [6]:
import shutil


def _safe_persist_path(raw_path: str) -> str:
    """Ensure raw_path resolves to a subdirectory of cwd. Prevents path traversal."""
    cwd = Path.cwd().resolve()
    resolved = (cwd / raw_path).resolve()
    if not resolved.is_relative_to(cwd):
        raise ValueError(
            f"Path '{raw_path}' resolves outside the working directory ('{resolved}'). "
            f"Refusing to delete or write outside '{cwd}'."
        )
    return str(resolved)


def build_vectorstore(chunks: List[Dict], persist_dir: str, include_image_summaries: bool = True):
    """Build vector store, optionally including image summaries.
    
    Args:
        chunks: List of document chunks
        persist_dir: Base directory for persistence
        include_image_summaries: If True, include image summaries; if False, text-only
    """
    
    suffix = "_with_summaries" if include_image_summaries else "_text_only"
    full_path = _safe_persist_path(persist_dir + suffix)

    # Remove stale DB to avoid version-mismatch panics and duplicate documents
    if os.path.exists(full_path):
        shutil.rmtree(full_path)
    
    # Create vectorstore
    vectorstore = Chroma(
        collection_name=f"nlm_report{suffix}",
        embedding_function=embeddings,
        persist_directory=full_path
    )
    
    # Filter chunks if needed
    filtered_chunks = chunks if include_image_summaries else [c for c in chunks if c['type'] == 'text']
    
    # Create documents
    documents = [
        Document(
            page_content=chunk['text'],
            metadata={
                **chunk['metadata'],
                'page': chunk['page'],
                'chunk_type': chunk['type']
            }
        )
        for chunk in filtered_chunks
    ]
    
    print(f"Creating vectorstore with {len(documents)} documents (include_image_summaries={include_image_summaries})...")
    vectorstore.add_documents(documents)
    
    print(f"Vectorstore created with suffix '{suffix}'")
    return vectorstore

def build_multimodal_retriever(chunks: List[Dict], persist_dir: str):
    """Build MultiVectorRetriever with text and actual images for Approach 2.
    
    Uses Qwen3 embeddings for text summaries (retrieval).
    Stores base64 images in docstore (sent to vision LLM during query).
    """
    full_path = _safe_persist_path(persist_dir + "_multimodal")

    # Remove stale DB to avoid version-mismatch panics and duplicate documents
    if os.path.exists(full_path):
        shutil.rmtree(full_path)
    
    # Create vectorstore for summaries/text (used for retrieval)
    vectorstore = Chroma(
        collection_name="nlm_report_multimodal",
        embedding_function=embeddings,
        persist_directory=full_path
    )
    
    # Create docstore for raw content (base64 images and text)
    docstore = InMemoryStore()
    id_key = "doc_id"
    
    # Create retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key
    )
    
    # Process chunks and add to retriever
    doc_ids = []
    summary_docs = []
    raw_contents = []
    
    for chunk in chunks:
        doc_id = str(uuid.uuid4())
        doc_ids.append(doc_id)
        
        if chunk['type'] == 'image_summary':
            # For images: summary goes to vectorstore, base64 goes to docstore
            summary_docs.append(
                Document(
                    page_content=chunk['text'],
                    metadata={
                        id_key: doc_id,
                        'type': 'image',
                        'page': chunk['page'],
                        'filename': chunk['metadata'].get('image_filename')
                    }
                )
            )
            
            # Encode image to base64 for docstore
            image_path = chunk['metadata']['image_path']
            with open(image_path, 'rb') as f:
                base64_image = base64.b64encode(f.read()).decode('utf-8')
            raw_contents.append(base64_image)
        else:
            # For text: same content in both vectorstore and docstore
            summary_docs.append(
                Document(
                    page_content=chunk['text'],
                    metadata={
                        id_key: doc_id,
                        'type': 'text',
                        'page': chunk['page'],
                        'source': chunk['metadata'].get('source', 'document_text')
                    }
                )
            )
            raw_contents.append(chunk['text'])
    
    print(f"Creating MultiVectorRetriever with {len(chunks)} documents...")
    print(f"  - TextNodes: {sum(1 for c in chunks if c['type'] == 'text')}")
    print(f"  - ImageNodes: {sum(1 for c in chunks if c['type'] == 'image_summary')}")
    
    # Add summaries to vectorstore
    retriever.vectorstore.add_documents(summary_docs)
    
    # Add raw content to docstore
    retriever.docstore.mset(list(zip(doc_ids, raw_contents)))
    
    print(f"MultiVectorRetriever created")
    return retriever

# Build all three stores
print("=" * 80)
text_only_vectorstore = build_vectorstore(chunks, CHROMA_DB_DIR, include_image_summaries=False)

print("\n" + "=" * 80)
summaries_vectorstore = build_vectorstore(chunks, CHROMA_DB_DIR, include_image_summaries=True)

print("=" * 80)
multimodal_retriever = build_multimodal_retriever(chunks, CHROMA_DB_DIR)

KeyboardInterrupt: 

### **Baseline - Text-Only Query (No Images)**

In [ ]:
def query_text_rag(vectorstore: Chroma, query: str, top_k: int = 5) -> str:
    """Query RAG system with text retriever (works for both text-only and text+summaries).
    
    Args:
        vectorstore: Chroma vectorstore (can be text-only or with image summaries)
        query: User query
        top_k: Number of documents to retrieve
        
    Returns:
        Response from the LLM
    """
    
    # Create LLM
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
    
    # Build simple RAG chain with LCEL
    retriever = vectorstore.as_retriever(search_kwargs={'k': top_k})
    
    # Create prompt template
    template = """Answer the question based only on the following context:
<context>
{context}
</context>
Your response should include the answer to the question without mention of the context.
Do not use your internal knowledge to answer the question.

Question: {question}
Answer:"""
    
    prompt = ChatPromptTemplate.from_template(template)
    
    # Build chain
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # Execute query
    response = rag_chain.invoke(query)

    return response

In [ ]:
query = "What was the growth of GenBank Base Pairs between Pubmed and Pubmed Central?"

In [ ]:
baseline_response = query_text_rag(text_only_vectorstore, query)
print(baseline_response)

In [ ]:
# Test with image summaries approach
img_summary_response = query_text_rag(summaries_vectorstore, query)
print(img_summary_response)

In [ ]:
def query_with_actual_images(retriever: MultiVectorRetriever, query: str, top_k: int = 5):
    """Query using MultiVectorRetriever with actual images sent to a VLM."""
        
    # Set retriever parameters
    retriever.search_kwargs = {'k': top_k}
    
    # Get summary documents from vectorstore to access metadata
    summary_docs = retriever.vectorstore.similarity_search(query, k=top_k)
    
    # Look up actual content from docstore using doc_ids
    doc_ids = [doc.metadata.get('doc_id') for doc in summary_docs]
    raw_contents = retriever.docstore.mget(doc_ids)
    
    # Helper function to split into images and text using metadata from summary_docs
    def split_image_text_types(summary_docs, raw_contents):
        """Separate retrieved content into images and text using summary doc metadata."""
        images = []
        texts = []
        
        for doc, content in zip(summary_docs, raw_contents):
            if content is None:
                continue
            if doc.metadata.get('type') == 'image':
                # content is base64 image string
                images.append(content)
            else:
                # content is text string
                texts.append(content)
        
        return {"images": images, "texts": texts}
    
    # Split retrieved content
    context = split_image_text_types(summary_docs, raw_contents)
    
    # Build message content for vision LLM
    content = [
        {
            "type": "text",
            "text": f"""
Answer the question based on the provided context and images.
Your response should include the answer to the question without mention of the context.
Do not use your internal knowledge to answer the question.
<context>
{'\n'.join(context['texts'])}
</context>

Question: {query}
Answer:"""
        }
    ]
    
    # Add images
    for img_base64 in context['images']:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{img_base64}"
            }
        })
    
    # Create vision-capable LLM and invoke
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
    response = llm.invoke([HumanMessage(content=content)])
    response_text = response.content
        
    return response_text

full_img_response = query_with_actual_images(multimodal_retriever, query)
print(full_img_response)